# Continue: measure scalable retrieval
The uploaded profile contains 2,206,821 training queries and 10,320,219 training targets. This notebook reuses your extracted dataset and preserves the master notebook. **Run all** performs relational auditing, constructs an experimental lexical index over the full training target pool, measures candidate recall on 256 fitting queries, and downloads a review report. It does not train or access the final holdout.

Use a CPU runtime. Initial normalization and indexing can take substantial time; progress and checkpoints are saved. Reconnect and run again to resume. This is a feasibility experiment, not a measured accuracy improvement. Check your actual remaining Google Drive quota: the earlier 79 GB figure described runtime disk space, not account quota. The normalized database and index need additional space beyond the extracted TSVs.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
import os, json, subprocess, sys
ROOT=Path('/content/drive/MyDrive/AmazonML2026')
location=ROOT/'data/raw/dataset_location.json'
assert location.is_file(), f'Missing {location}; finish dataset extraction in the master notebook first.'
DATASET=Path(json.loads(location.read_text())['dataset'])
assert (DATASET/'train/train_ground_truth.tsv').is_file(), f'Dataset not found: {DATASET}'
print('Reusing:', DATASET)


## Load the continuation code
A separate local checkout protects edits in the master checkout. The first run saves a continuation-specific commit; reconnects use that same revision for compatible checkpoints. It does not use or change the master notebook's saved commit.

In [ ]:
REPO=Path('/content/Amazon-ML-C-26-continuation')
SESSION=ROOT/'lexical_screen_code_commit.json'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/SanskariXD/Amazon-ML-C-26.git',str(REPO)],check=True)
if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
    raise RuntimeError('Continuation checkout contains edits; save them before switching revisions.')
pin=json.loads(SESSION.read_text())['commit'] if SESSION.exists() else ''
subprocess.run(['git','-C',str(REPO),'fetch','origin',pin or 'main'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',pin or 'origin/main'],check=True)
os.chdir(REPO)
assert Path('configs/colab_screen.json').is_file(), 'The pinned revision lacks the continuation config.'
commit=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
SESSION.write_text(json.dumps({'commit':commit}))
print('Continuation commit:',commit)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
os.environ['ER_LOCAL_CACHE']='/content/amazon_ml_cache'
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'


## Audit and benchmark
Country blocking is permitted only if the relational audit finds zero cross-country or missing-country labelled pairs. Other integrity failures also stop the run. All S2/S3 training targets are indexed; individual keys with more than 2,048 postings are omitted and the report records that count. No positive candidates are injected using labels.

The first 256 fitting queries selected by a seeded hash are a preliminary screen. The report includes pair recall, complete-entity recall, country breakdowns, storage, and an approximate retrieval-time projection. It cannot estimate France accuracy because France has no training labels. No benchmark threshold automatically promotes the method.

In [ ]:
CONFIG='configs/colab_screen.json'
def run(stage):
    subprocess.run([sys.executable,'-m','src.pipeline','--dataset',str(DATASET),'--work',str(ROOT/'runs'),'--config',CONFIG,'--stage',stage],check=True)
    return Path(json.loads((ROOT/'runs/latest_run.json').read_text())['work'])
run('audit')
WORK=run('benchmark')
print((WORK/'experiments/blocking_benchmark.json').read_text())


## Download the report
Upload the new `review_report.zip` to the conversation. The next decision is whether recall and measured cost justify fitting the 5,000-query screening model, adjusting key limits, or testing another retriever. Large inference and the final holdout remain deferred until the method is selected.

In [ ]:
WORK=run('report')
files.download(str(WORK/'review_report.zip'))
